In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
            .master("local[*]") \
            .appName('test') \
            .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/07/06 19:04:12 WARN Utils: Your hostname, SRCIND-21BQ9G3 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/07/06 19:04:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/06 19:04:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load the JSON data
ad_campaigns_data = './ad_campaigns_data.json'
user_profile_data = './user_profile_data.json'
store_data = './store_data.json'


# Define the schema
schema_campaigns = T.StructType([
    T.StructField("campaign_id", T.StringType(), True),
    T.StructField("campaign_name", T.StringType(), True),
    T.StructField("campaign_country", T.StringType(), True),
    T.StructField("os_type", T.StringType(), True),
    T.StructField("device_type", T.StringType(), True),
    T.StructField("place_id", T.StringType(), True),
    T.StructField("user_id", T.StringType(), True),
    T.StructField("event_type", T.StringType(), True),
    T.StructField("event_time", T.TimestampType(), True)
])


schema_users = T.StructType([
    T.StructField("user_id", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("gender", T.StringType(), True),
    T.StructField("age_group", T.StringType(), True),
    T.StructField("category", T.ArrayType(T.StringType()), True)
])


schema_stores = T.StructType([
    T.StructField("store_name", T.StringType(), True),
    T.StructField("place_ids", T.ArrayType(T.StringType()), True)
])

In [6]:
df_campaigns = spark.read.format('json').schema(schema_campaigns).load(ad_campaigns_data)
df_users = spark.read.format('json').schema(schema_users).load(user_profile_data)
df_stores = spark.read.format('json').schema(schema_stores).load(store_data)

In [9]:
# Show the dataframes
df_campaigns.show(5)

df_users.show(5)

df_stores.show(5)

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 18:40:05|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 18:39:04|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|BADGBA-12|   5747421465445443|  video ad|2018-10-12 18:40:10|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|CASSBB-11|1864374214654454132|     click|2018-10-12 18:40:12|
+-----------+--------------------+----------------+-------+----------

In [10]:
df_campaigns.printSchema()
df_users.printSchema()
df_stores.printSchema()

root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_country: string (nullable = true)
 |-- os_type: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- place_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

root
 |-- user_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)

root
 |-- store_name: string (nullable = true)
 |-- place_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [14]:
# Extract date and hour from event_time
df_campaigns = df_campaigns.withColumn("date", F.to_date("event_time"))
df_campaigns = df_campaigns.withColumn("hour", F.hour("event_time"))
df_campaigns.show(5)

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|      date|hour|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 18:40:05|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 18:39:04|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|BADGBA-12|   5747421465445443|  video ad|2018-10-12 18:40:10|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|CASSBB-11|1864374214654454132|     

Q1. Analyse data for each campaign_id, date, hour, os_type & value to get all the
events with counts

In [21]:
q1_result = (
    df_campaigns
    .groupBy("campaign_id", "date", "hour", "os_type", "event_type")
    .agg(F.count("*").alias("event_count"))
)
q1_result.show()

+-----------+----------+----+-------+----------+-----------+
|campaign_id|      date|hour|os_type|event_type|event_count|
+-----------+----------+----+-------+----------+-----------+
|    ABCDFAE|2018-10-12|  18|android|impression|          1|
|    ABCDFAE|2018-10-12|  18|android|     click|          1|
|    ABCDFAE|2018-10-12|  18|    ios|impression|          1|
|    ABCDFAE|2018-10-12|  18|android|  video ad|          1|
+-----------+----------+----+-------+----------+-----------+



In [ ]:
q1_result = (
    df_campaigns
    .groupBy("campaign_id", "date", "hour", "os_type", "event_type")
    .agg(F.count("*").alias("event_count"))
    .groupBy("campaign_id", "date", "hour", "os_type")
    .pivot("event_type")
    .agg(F.sum("event_count"))
    .fillna(0)
)
q1_result.show()

+-----------+----------+----+-------+-----+----------+--------+
|campaign_id|      date|hour|os_type|click|impression|video ad|
+-----------+----------+----+-------+-----+----------+--------+
|    ABCDFAE|2018-10-12|  18|android|    1|         1|       1|
|    ABCDFAE|2018-10-12|  18|    ios|    0|         1|       0|
+-----------+----------+----+-------+-----+----------+--------+



In [28]:
q1_result = (
    df_campaigns
    .groupBy("campaign_id", "date", "hour", "os_type", "event_type")
    .agg(F.count("*").alias("event_count"))
    .groupBy("campaign_id", "date", "hour", "os_type")
    .pivot("event_type")
    .agg(F.sum("event_count"))
    .fillna(0)
    .select(
        "campaign_id",
        "date",
        "hour",
        "os_type",
        F.struct(
            F.col("impression").alias("impression"),
            F.col("click").alias("click"),
            F.col("video ad").alias("video ad")
        ).alias("event")
    )
)
q1_result.show(truncate=False)

+-----------+----------+----+-------+---------+
|campaign_id|date      |hour|os_type|event    |
+-----------+----------+----+-------+---------+
|ABCDFAE    |2018-10-12|18  |android|{1, 1, 1}|
|ABCDFAE    |2018-10-12|18  |ios    |{1, 0, 0}|
+-----------+----------+----+-------+---------+



In [29]:
q1_result.write.mode("overwrite").json("./results/q1_result.json")

Q2.Analyse data for each campaign_id, date, hour, store_name & value to get all the
events with counts

In [31]:
df_campaigns.show(5)
df_stores.show(5)

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|      date|hour|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 18:40:05|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 18:39:04|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|BADGBA-12|   5747421465445443|  video ad|2018-10-12 18:40:10|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|CASSBB-11|1864374214654454132|     

In [33]:
df_campaigns.join(df_stores, F.array_contains(df_stores.place_ids, df_campaigns.place_id)).show(5)

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+-------------+--------------------+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|      date|hour|   store_name|           place_ids|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+-------------+--------------------+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 18:40:05|2018-10-12|  18|     McDonald|[CASSBB-11, CADGB...|
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 18:40:05|2018-10-12|  18|   BurgerKing|         [CASSBB-11]|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13

In [38]:
q2_result = (
    df_campaigns
    .join(df_stores, F.array_contains(df_stores.place_ids, df_campaigns.place_id))
    .groupBy("campaign_id", "date", "hour", "store_name", "event_type")
    .agg(F.count("*").alias("event_count"))
)
q2_result.show(truncate=False)

+-----------+----------+----+-------------+----------+-----------+
|campaign_id|date      |hour|store_name   |event_type|event_count|
+-----------+----------+----+-------------+----------+-----------+
|ABCDFAE    |2018-10-12|18  |BurgerKing   |impression|1          |
|ABCDFAE    |2018-10-12|18  |shoppers stop|video ad  |1          |
|ABCDFAE    |2018-10-12|18  |McDonald     |click     |1          |
|ABCDFAE    |2018-10-12|18  |BurgerKing   |click     |1          |
|ABCDFAE    |2018-10-12|18  |McDonald     |impression|2          |
+-----------+----------+----+-------------+----------+-----------+



In [43]:
q2_result = (
    df_campaigns
    .join(df_stores, F.array_contains(df_stores.place_ids, df_campaigns.place_id))
    .groupBy("campaign_id", "date", "hour", "store_name", "event_type")
    .agg(F.count("*").alias("event_count"))

    .groupBy("campaign_id", "date", "hour", "store_name")
    .pivot("event_type")
    .agg(F.sum("event_count"))
    .fillna(0)
    .select(
        "campaign_id",
        "date",
        "hour",
        "store_name",
        F.struct(
            F.col("impression").alias("impression"),
            F.col("click").alias("click"),
            F.col("video ad").alias("video ad")
        ).alias("event")
    )
)
q2_result.show(truncate=False)

+-----------+----------+----+-------------+---------+
|campaign_id|date      |hour|store_name   |event    |
+-----------+----------+----+-------------+---------+
|ABCDFAE    |2018-10-12|18  |McDonald     |{2, 1, 0}|
|ABCDFAE    |2018-10-12|18  |shoppers stop|{0, 0, 1}|
|ABCDFAE    |2018-10-12|18  |BurgerKing   |{1, 1, 0}|
+-----------+----------+----+-------------+---------+



In [47]:
q2_result.write.mode("overwrite").json("./results/q2_result.json")

Q3.Analyse data for each campaign_id, date, hour, gender_type & value to get all the
events with counts

In [58]:
df_campaigns.show(5)
df_users.show(5)

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|      date|hour|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 18:40:05|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 18:39:04|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|BADGBA-12|   5747421465445443|  video ad|2018-10-12 18:40:10|2018-10-12|  18|
|    ABCDFAE|Food category tar...|             USA|android|    SAMSUNG|CASSBB-11|1864374214654454132|     

In [59]:
df_campaigns.join(df_users, df_campaigns.user_id == df_users.user_id, how='left').show(5)

+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+-------------------+-------+------+---------+--------------------+
|campaign_id|       campaign_name|campaign_country|os_type|device_type| place_id|            user_id|event_type|         event_time|      date|hour|            user_id|country|gender|age_group|            category|
+-----------+--------------------+----------------+-------+-----------+---------+-------------------+----------+-------------------+----------+----+-------------------+-------+------+---------+--------------------+
|    ABCDFAE|Food category tar...|             USA|    ios|      apple|CASSBB-11|1264374214654454321|impression|2018-10-12 18:40:05|2018-10-12|  18|1264374214654454321|    USA|  male|    18-25|  [shopper, student]|
|    ABCDFAE|Food category tar...|             USA|android|   MOTOROLA|CADGBD-13|1674374214654454321|impression|2018-10-12 18:39:04|2018-10-

In [60]:
q3_result = (
    df_campaigns.join(df_users, df_campaigns.user_id == df_users.user_id, how='left')
    .groupBy("campaign_id", "date", "hour", "gender", "event_type")
    .agg(F.count("*").alias("event_count"))

    .groupBy("campaign_id", "date", "hour", "gender")
    .pivot("event_type")
    .agg(F.sum("event_count"))
    .fillna(0)
)

q3_result.show(5)

+-----------+----------+----+------+-----+----------+--------+
|campaign_id|      date|hour|gender|click|impression|video ad|
+-----------+----------+----+------+-----+----------+--------+
|    ABCDFAE|2018-10-12|  18|  male|    1|         1|       1|
|    ABCDFAE|2018-10-12|  18|female|    0|         1|       0|
+-----------+----------+----+------+-----+----------+--------+



In [64]:
q3_result = (
    df_campaigns.join(df_users, df_campaigns.user_id == df_users.user_id, how='left')
    .groupBy("campaign_id", "date", "hour", "gender", "event_type")
    .agg(F.count("*").alias("event_count"))

    .groupBy("campaign_id", "date", "hour", "gender")
    .pivot("event_type")
    .agg(F.sum("event_count"))
    .fillna(0)
    .select(
        "campaign_id",
        "date",
        "hour",
        "gender",
        F.struct(
            F.col("impression").alias("impression"),
            F.col("click").alias("click"),
            F.col("video ad").alias("video ad")
        ).alias("event")
    )
)

q3_result.show(5)

+-----------+----------+----+------+---------+
|campaign_id|      date|hour|gender|    event|
+-----------+----------+----+------+---------+
|    ABCDFAE|2018-10-12|  18|  male|{1, 1, 1}|
|    ABCDFAE|2018-10-12|  18|female|{1, 0, 0}|
+-----------+----------+----+------+---------+



In [62]:
q3_result.write.mode("overwrite").json("./results/q3_result.json")

In [65]:
spark.stop()
# End of file